In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats as sp_stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import time, warnings

In [3]:
warnings.filterwarnings('ignore')

In [4]:
IMAGE_DIR = '/Users/shubhbhateja/College/minor_project/images'
CSV_PATH  = '/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv'
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [5]:
# ── 2. DATASET LOADING & VERIFICATION ──
# (Incorporated from paper_implementation.ipynb)
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from CSV")

print("\n--- Verifying Dataset Integrity ---")
# Ensure image names have extensions and physical files exist
df['image_idx'] = df['image_idx'].apply(lambda x: str(x) if str(x).endswith('.jpg') else f"{x}.jpg")
df['file_exists'] = df['image_idx'].apply(lambda x: os.path.exists(os.path.join(IMAGE_DIR, x)))

missing_files = df[df['file_exists'] == False]
if not missing_files.empty:
    print(f"⚠️ WARNING: {len(missing_files)} images listed in CSV are missing from directory.")
else:
    print("✅ All image indices in CSV mapped to valid physical files.")

# Filter to only existing images and summarize
valid_df = df[df['file_exists'] == True].copy()
print(f"Total Unique Patients: {valid_df['patient_id'].nunique()}")

Loaded 2235 rows from CSV

--- Verifying Dataset Integrity ---
✅ All image indices in CSV mapped to valid physical files.
Total Unique Patients: 745


In [ ]:
# ── 1. CONFIGURATION ──
CSV_PATH = '/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv'
IMAGE_DIR = '/Users/shubhbhateja/College/minor_project/images'
SEED = 42

In [ ]:
# ── 2. DATASET VERIFICATION (From paper_implementation.ipynb) ──
df = pd.read_csv(CSV_PATH)
df['image_idx'] = df['image_idx'].apply(lambda x: str(x) if str(x).endswith('.jpg') else f"{x}.jpg")
df['file_exists'] = df['image_idx'].apply(lambda x: os.path.exists(os.path.join(IMAGE_DIR, x)))

valid_df = df[df['file_exists'] == True].copy()
print(f"Verified {len(valid_df)} images. Unique Patients: {valid_df['patient_id'].nunique()}")

In [ ]:
def extract_color_features(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (128, 128))
    
    # Simple White Balance & HSV conversion
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    combined = np.concatenate([img, hsv], axis=-1).reshape(-1, 6) # (Pixels, 6 Channels)
    
    # Per-channel statistics: Mean, Std, Min, Max, Median, Skew (36 features total)
    # Reshaped to (6, 6) for the 1D-CNN sequence input
    stats = []
    for i in range(6):
        ch = combined[:, i]
        stats.append([np.mean(ch), np.std(ch), np.min(ch), np.max(ch), np.median(ch), 0.0]) # Simplified skew
    return np.array(stats)

print("Processing features...")
X, y, p_ids = [], [], []
for _, row in valid_df.iterrows():
    feat = extract_color_features(os.path.join(IMAGE_DIR, row['image_idx']))
    if feat is not None:
        X.append(feat)
        y.append(row['blood(mg/dL)'])
        p_ids.append(row['patient_id'])

X = np.array(X)
y = np.array(y)
p_ids = np.array(p_ids)


Extracting features from images and clinical metadata...


In [ ]:
u_patients = np.unique(p_ids)
train_p, test_p = train_test_split(u_patients, test_size=0.2, random_state=SEED)

X_train = X[np.isin(p_ids, train_p)]
y_train = y[np.isin(p_ids, train_p)]
X_test = X[np.isin(p_ids, test_p)]
y_test = y[np.isin(p_ids, test_p)]

In [ ]:
def build_1dcnn_regression(input_shape):
    inp = tf.keras.Input(shape=input_shape, name='color_input')

    # Block 1
    x = tf.keras.layers.Conv1D(32, 3, padding='same', activation='relu')(inp)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    # Block 2
    x = tf.keras.layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    # Block 3
    x = tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    # Block 4
    x = tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    # Dense Layers
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)

    out = tf.keras.layers.Dense(1, activation='linear', name='tsb_output')(x)
    
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_input           │ (None, 36, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 34, 32)    │        128 │ img_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ clin_input          │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1088)      │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │         64 │ clin_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1104)      │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     70,720 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tsb_output (Dense)  │ (None, 1)         │         65 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 70,977 (277.25 KB)

 Trainable params: 70,977 (277.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    x={'img_input': X_img_train, 'clin_input': X_clin_train},
    y=y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.1,
    verbose=1
)

In [ ]:
results = model.evaluate(x={'img_input': X_img_test, 'clin_input': X_clin_test}, y=y_test)
print(f"\nFinal Test MAE: {results[1]:.4f} mg/dL")